# Módulo 02 · Aula 1: Retrievers e filtros

**Duração:** 1 h

## Objetivos de aprendizagem

Ao final desta aula você será capaz de:

- transformar o índice FAISS em um **retriever** (`as_retriever`) e usá-lo como qualquer Runnable do LCEL;
- escolher o número de trechos **`k`** conhecendo o custo de errar para mais ou para menos;
- ler as **notas** de `similarity_search_with_score` e explicar por que um **limiar fixo** de score é frágil;
- usar **MMR** (`max_marginal_relevance_search`) para trocar trechos repetidos por trechos diversos;
- aplicar **filtros por metadados** como controle de **atualidade** (documentos arquivados) e de **acesso**
  (documentos internos), com `make_filter` e `CUSTOMERS_ONLY` de `src/rag/retrieval.py`.

## Pré-requisitos

- Módulo 01, aula 2.3 (`01_03_vector_search`): embeddings, FAISS e `similarity_search`.
- Módulo 01, aulas 2.4 e 5 (`01_04_document_loading` e `01_05_chunking_and_indexing`): os metadados `source`, `status` e `audience` de cada
  trecho e o índice principal em `data/faiss_index` (321 trechos).

## O que vamos construir

A etapa de **recuperação** do assistente: dada uma pergunta, devolver os trechos certos, e somente os que o cliente
pode ver.

```mermaid
flowchart LR
    Q([pergunta]) --> R[retriever<br/>k, MMR]
    I[(data/faiss_index)] --> R
    R --> F{filtro de<br/>metadados}
    F -->|status = active<br/>audience = customers| T[k trechos]
    F -.arquivado ou interno.-> X[descartado]
```

> Custo desta aula: nenhuma chamada ao modelo de chat, só embeddings de cerca de 12 consultas curtas. Quando a mesma
> pergunta é usada várias vezes, calculamos o vetor uma vez e reaproveitamos.

In [1]:
# Os notebooks rodam a partir da raiz do projeto: é lá que ficam o .env, a pasta data/ e o pacote src/.
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Chat e embeddings vêm do Azure OpenAI configurado no .env (veja src/config.py).
from src.config import settings

## 1. Da busca ao retriever

Na aula 1.3 buscamos com `index.similarity_search(pergunta, k=4)`, uma API específica do FAISS. O LangChain define
uma interface comum para "quem recupera documentos", o **retriever**: entra uma string, sai uma `list[Document]`. Todo
vector store vira retriever com `as_retriever()`, e por ser um **Runnable** ele ganha `invoke`, `batch`, `stream` e
composição com `|`.

Carregamos o índice principal (o FAISS vem sempre de `src.rag.vectorstore`, o único arquivo do projeto que depende do
`langchain-community`):

In [2]:
from src.rag.vectorstore import index_exists, load_index, stored_chunks

if not index_exists():
    from src.rag.ingestion import index_folder

    index_folder()

index = load_index()
all_chunks = list(stored_chunks(index).values())
print(f"{len(all_chunks)} trechos no índice")


def show(docs, scores=None):
    """Imprime posição, (nota), fonte, página ou seção e o começo de cada trecho."""
    for position, doc in enumerate(docs, start=1):
        md = doc.metadata
        where = f"página {md['page']}" if md.get("page") else (md.get("section") or "")
        score = f"{scores[position - 1]:.3f}  " if scores is not None else ""
        preview = doc.page_content.split("\n", 1)[-1][:60].replace("\n", " ")
        print(f"{position}. {score}{md['source']:<40} {where[:22]:<22} {preview}")

2026-09-27 22:31:46 | INFO     | src.models | Inicializando embeddings | provedor=azure | deployment=text-embedding-3-large


321 trechos no índice


In [3]:
from langchain_core.runnables import Runnable, RunnableLambda

retriever = index.as_retriever(search_kwargs={"k": 4})
show(retriever.invoke("Em quantos dias posso trocar um produto?"))

# Por ser um Runnable, o retriever encaixa em uma chain LCEL: aqui, só as fontes, com duas perguntas em lote.
print("\nÉ um Runnable?", isinstance(retriever, Runnable))
sources_chain = retriever | RunnableLambda(lambda docs: [d.metadata["source"] for d in docs])
for sources in sources_chain.batch(["Como rastreio meu pedido?", "Esqueci minha senha"]):
    print(sources)

1. faq.csv                                                         Pergunta: Qual o prazo para trocar um produto? Resposta: Até
2. help_center/exchange_for_another_model.html                        Como trocar um produto por outra cor ou modelo  Mudou de ide
3. policies/exchange.md                     Resumo da regra        ## Resumo da regra   - Trocas podem ser solicitadas em **até
4. policies/exchange.md                     Direito de arrependime ## Direito de arrependimento e prazo ampliado   O Código de 

É um Runnable? True


['help_center/track_order.html', 'faq.csv', 'policies/shipping.md', 'faq.csv']
['faq.csv', 'help_center/reset_password.html', 'help_center/reset_password.html', 'help_center/reset_password.html']


**Observe:** `as_retriever` devolve um `VectorStoreRetriever`, que guarda **como** buscar (`search_type` e
`search_kwargs`). Quem o usa não sabe se por trás há FAISS, outro banco vetorial ou uma busca por palavras: é essa troca
que faremos na próxima aula, com o `HybridRetriever` do produto.

> 💡 **Dica:** `faq.csv` aparece com frequência. As perguntas frequentes são curtas e escritas como o cliente pergunta,
> por isso ficam muito perto das consultas no espaço vetorial. Isso vai importar quando medirmos a busca por "fonte
> esperada" (aula 2.2 e aula 3.4).

## 2. Quantos trechos? O parâmetro `k`

`k` é quantos trechos vão para o modelo. Não existe valor certo, existe um equilíbrio:

| `k` pequeno (1 a 2) | `k` grande (10 ou mais) |
|---|---|
| prompt curto, barato e rápido | mais chance de o trecho certo estar lá |
| basta a busca errar a ordem para perder a resposta | mais ruído e mais tokens, em toda pergunta |

A fonte certa da pergunta abaixo é `policies/warranty.md`. Calculamos o vetor da pergunta **uma vez** e reaproveitamos
com `similarity_search_by_vector` (sem nova chamada de embeddings):

In [4]:
from langchain_core.messages.utils import count_tokens_approximately

from src.models import get_embeddings

embeddings = get_embeddings()
question = "Quanto tempo de garantia tem um notebook comprado na TechNova?"
question_vector = embeddings.embed_query(question)

for k in (1, 2, 4, 8):
    docs = index.similarity_search_by_vector(question_vector, k=k)
    sources = [d.metadata["source"] for d in docs]
    tokens = count_tokens_approximately([d.page_content for d in docs])
    found = "sim" if "policies/warranty.md" in sources else "não"
    print(f"k={k}: política de garantia no contexto? {found:<3} | ~{tokens:>5} tokens de contexto")

k=1: política de garantia no contexto? não | ~  272 tokens de contexto
k=2: política de garantia no contexto? não | ~  456 tokens de contexto
k=4: política de garantia no contexto? sim | ~  785 tokens de contexto
k=8: política de garantia no contexto? sim | ~ 1520 tokens de contexto


**Observe:** com `k=1` ou `k=2` a política some: ela está em 3º lugar, atrás de um artigo da central de ajuda e do
manual do notebook. O projeto usa `k=4` (`RETRIEVAL_K` no `.env`): cabe folgado no contexto e dá margem para a busca
não ser perfeita. Vamos medir essa escolha na próxima aula e na aula 3.4.

## 3. As notas da busca: `similarity_search_with_score`

`similarity_search_with_score` devolve pares `(Document, nota)`. No FAISS do projeto a nota é uma **distância** (o
quadrado da distância euclidiana): **quanto menor, mais parecido**. Como os vetores do `text-embedding-3-large` têm
comprimento 1, `cosseno = 1 - distância / 2`.

In [5]:
results = index.similarity_search_with_score_by_vector(question_vector, k=4)
docs, distances = zip(*results)
show(docs, distances)

print("\nDistância -> cosseno:", [round(1 - float(d) / 2, 3) for d in distances])

1. 0.673  help_center/warranty_claim_step_by_step.html                        Como acionar a garantia: passo a passo  Todos os produtos ve
2. 0.675  manuals/nb-001_nova_air_14.pdf           página 5               problema não for resolvido nesse prazo, você escolhe entre a
3. 0.706  policies/warranty.md                                            # Política de Garantia   Todos os produtos vendidos pela Tec
4. 0.725  help_center/warranty_claim_step_by_step.html                        Se a assistência técnica não resolver o problema em até 30 d

Distância -> cosseno: [0.664, 0.663, 0.647, 0.637]


> ⚠️ Cada vector store tem a sua convenção. No FAISS com distância euclidiana, menor é melhor; em outros bancos a nota
> já é um cosseno ou um "relevance score" de 0 a 1, e maior é melhor. Confira antes de comparar notas.

## 4. Limiar de score, e por que um limiar fixo é frágil

A ideia é tentadora: "se a melhor distância for maior que X, a base não tem a resposta". Testamos quatro perguntas,
duas que a base responde e duas que ela **não** responde (a base de conhecimento não tem o manual do Nova Gamer 16 nem vagas de emprego), e aplicamos
um limiar fixo de 0,85:

In [6]:
THRESHOLD = 0.85
probe_questions = {
    "Qual o valor mínimo para ter frete grátis?": "responde",
    "Quanto dura a bateria do fone?": "responde",
    "Qual a placa de vídeo do Nova Gamer 16?": "NÃO responde",
    "A TechNova está contratando?": "NÃO responde",
}

for q, expected in probe_questions.items():
    results = index.similarity_search_with_score(q, k=4)
    distances = ", ".join(f"{s:.3f}" for _, s in results)
    kept = [doc.metadata["source"] for doc, distance in results if distance <= THRESHOLD]
    print(f"[{expected:<12}] {q}\n   distâncias: {distances}\n   até {THRESHOLD}: {len(kept)} trecho(s) {kept}")

[responde    ] Qual o valor mínimo para ter frete grátis?
   distâncias: 0.413, 0.640, 0.696, 0.710
   até 0.85: 4 trecho(s) ['faq.csv', 'policies/shipping_2024.md', 'policies/shipping_2024.md', 'policies/shipping.md']


[responde    ] Quanto dura a bateria do fone?
   distâncias: 0.708, 0.850, 0.893, 0.935
   até 0.85: 1 trecho(s) ['faq.csv']


[NÃO responde] Qual a placa de vídeo do Nova Gamer 16?
   distâncias: 0.884, 0.908, 0.914, 0.923
   até 0.85: 0 trecho(s) []


[NÃO responde] A TechNova está contratando?
   distâncias: 0.781, 0.849, 0.851, 0.895
   até 0.85: 2 trecho(s) ['help_center/support_channels_and_hours.html', 'help_center/support_channels_and_hours.html']


**Observe:** o limiar erra para os dois lados ao mesmo tempo.

- "Quanto dura a bateria do fone?" tem resposta, mas o **manual** do Nova Buds (0,89) fica de fora; sobra a FAQ.
- "A TechNova está contratando?" não tem resposta, mas dois trechos do artigo de canais de atendimento **passam**.
- O Nova Gamer 16 foi cortado por pouco: a melhor distância dele (0,88, o manual do **Nova Pro 16**) é **menor** que a
  do manual do Nova Buds. Um limiar que deixa o fone entrar deixa o notebook errado entrar também.
- No frete, passam dois trechos da política **arquivada**: o limiar não sabe nada sobre validade (seção 6).

| Motivo | Consequência |
|---|---|
| A distância mede **parecença**, não **se o trecho responde** | um trecho vizinho do assunto passa |
| A escala muda com a pergunta e com o modelo de embeddings | o número calibrado não se transfere |

> 💡 **Dica:** `as_retriever(search_type="similarity_score_threshold")` tem o mesmo problema. Use limiar só como trava
> grosseira, calibrada com perguntas de avaliação. Decidir "a base responde?" é trabalho da **geração** (aula 2.3) e da
> **avaliação de relevância** do RAG corretivo (aula 3.2).

## 5. Diversidade com MMR

Trechos do mesmo documento são **parecidos entre si** (sobreposição do chunking, páginas que repetem o assunto), e os
`k` lugares podem ser ocupados por quase o mesmo texto. O **MMR** (Maximal Marginal Relevance) escolhe os trechos um a
um, pegando a cada passo o candidato que maximiza

$$\text{MMR} = \lambda \cdot \text{sim}(\text{pergunta}, d) \; - \; (1 - \lambda) \cdot \max_{s \in \text{escolhidos}} \text{sim}(d, s)$$

ou seja: parecido com a pergunta, **mas diferente** do que já foi escolhido.

| Parâmetro | Significado |
|---|---|
| `fetch_k` | quantos candidatos buscar antes de escolher (o MMR só reordena esses) |
| `k` | quantos devolver |
| `lambda_mult` | de 0 a 1: 1 = só relevância (igual à busca comum); 0 = só diversidade |

In [7]:
mmr_question = "Como funciona a garantia do Notebook Nova Air 14?"
mmr_vector = embeddings.embed_query(mmr_question)

print("Busca comum:")
show(index.similarity_search_by_vector(mmr_vector, k=4))

print("\nMMR (fetch_k=40, lambda_mult=0.5):")
show(index.max_marginal_relevance_search_by_vector(mmr_vector, k=4, fetch_k=40, lambda_mult=0.5))

ranking = [d.metadata["source"] for d in index.similarity_search_by_vector(mmr_vector, k=40)]
print("\nPosição da política de garantia na busca comum:", ranking.index("policies/warranty.md") + 1)

Busca comum:
1. manuals/nb-001_nova_air_14.pdf           página 5               - Danos causados por queda ou impacto. - Contato com líquido
2. manuals/nb-001_nova_air_14.pdf           página 5               problema não for resolvido nesse prazo, você escolhe entre a
3. manuals/nb-002_nova_pro_16.pdf           página 5               A garantia não cobre danos por queda, contato com líquidos e
4. manuals/nb-001_nova_air_14.pdf           página 5               TechNova | Notebook Nova Air 14 (NB-001) | Manual do usuário

MMR (fetch_k=40, lambda_mult=0.5):
1. manuals/nb-001_nova_air_14.pdf           página 5               - Danos causados por queda ou impacto. - Contato com líquido
2. manuals/nb-002_nova_pro_16.pdf           página 2               - Lateral direita: USB-A 3.2, leitor de cartão SD e porta de
3. manuals/nb-001_nova_air_14.pdf           página 1               TechNova Notebook Nova Air 14 SKU: NB-001 Manual do usuário 
4. policies/warranty.md                                

**Observe:** na busca comum, três dos quatro lugares são da **mesma página** do manual do Nova Air 14, e a política
oficial de garantia fica de fora. Com o MMR ela entra (estava em 31º lugar) e sai a repetição. Mas o 2º trecho, a página
2 do manual do Nova Pro 16 (conectores e peso), entrou só por ser **diferente**. E o `fetch_k` alto (40) importa: o MMR
só escolhe entre os candidatos buscados.

Variando `lambda_mult` e contando as fontes distintas (arquivo e página):

In [8]:
for lambda_mult in (1.0, 0.75, 0.5, 0.25):
    docs = index.max_marginal_relevance_search_by_vector(mmr_vector, k=4, fetch_k=40, lambda_mult=lambda_mult)
    labels = [d.metadata["source"].split("/")[-1] + (f"#p{d.metadata['page']}" if d.metadata.get("page") else "") for d in docs]
    print(f"lambda_mult={lambda_mult:<4}: {len(set(labels))} fontes distintas (arquivo#página) | {labels}")

lambda_mult=1.0 : 2 fontes distintas (arquivo#página) | ['nb-001_nova_air_14.pdf#p5', 'nb-001_nova_air_14.pdf#p5', 'nb-002_nova_pro_16.pdf#p5', 'nb-001_nova_air_14.pdf#p5']
lambda_mult=0.75: 2 fontes distintas (arquivo#página) | ['nb-001_nova_air_14.pdf#p5', 'nb-001_nova_air_14.pdf#p5', 'nb-001_nova_air_14.pdf#p2', 'nb-001_nova_air_14.pdf#p5']
lambda_mult=0.5 : 4 fontes distintas (arquivo#página) | ['nb-001_nova_air_14.pdf#p5', 'nb-002_nova_pro_16.pdf#p2', 'nb-001_nova_air_14.pdf#p1', 'warranty.md']
lambda_mult=0.25: 4 fontes distintas (arquivo#página) | ['nb-001_nova_air_14.pdf#p5', 'nb-002_nova_pro_16.pdf#p3', 'warranty.md', 'warranty_claim_step_by_step.html']


> ⚠️ Diversidade tem preço: com `lambda_mult` baixo, o MMR começa a trazer trechos **diferentes, mas irrelevantes**,
> só porque são diferentes. Comece em 0,5 a 0,7 e confira com perguntas reais.

No retriever, é só trocar o `search_type`:

```python
mmr_retriever = index.as_retriever(search_type="mmr", search_kwargs={"k": 4, "fetch_k": 40, "lambda_mult": 0.5})
```

O produto não usa MMR por padrão: a busca híbrida da próxima aula já mistura candidatos de duas buscas diferentes, o
que traz parte dessa diversidade. O MMR é um botão que vale conhecer para bases com muita repetição.

## 6. Filtros por metadados

Cada trecho carrega os metadados do documento de origem (aula 1.4). Dois deles existem justamente para a busca:

| Metadado | Valores | Para quê |
|---|---|---|
| `status` | `active`, `archived` | **atualidade**: documento em vigor ou substituído |
| `audience` | `customers`, `internal` | **acesso**: pode ser mostrado ao cliente ou não |

In [9]:
from collections import Counter

counts = Counter((c.metadata["status"], c.metadata["audience"]) for c in all_chunks)
for (status, audience), n in counts.most_common():
    sources = sorted({c.metadata["source"] for c in all_chunks if (c.metadata["status"], c.metadata["audience"]) == (status, audience)})
    print(f"status={status:<9} audience={audience:<10} {n:>3} trechos  {sources if len(sources) < 3 else ''}")

status=active    audience=customers  298 trechos  
status=archived  audience=customers   13 trechos  ['policies/shipping_2024.md']
status=active    audience=internal    10 trechos  ['internal/refund_approval_procedure.md']


### 6.1 Atualidade: a política de frete arquivada

A política de frete de 2024 (`policies/shipping_2024.md`) foi substituída, mas continua no índice (é comum manter o
histórico). O FAISS aceita o parâmetro `filter`: um dicionário (igualdade simples, ex.: `{"status": "active"}`) ou uma
**função** que recebe os metadados e devolve `True`/`False`. O produto cria essas funções com `make_filter`. Sem filtro
e com filtro:

In [10]:
from src.rag.retrieval import CUSTOMERS_ONLY, make_filter

shipping_question = "Qual o valor mínimo para ter frete grátis?"
shipping_vector = embeddings.embed_query(shipping_question)

docs = index.similarity_search_by_vector(shipping_vector, k=4)
print("Sem filtro:")
show(docs)
for doc in docs:
    if doc.metadata.get("section") == "Frete grátis":
        rule = [line for line in doc.page_content.splitlines() if "Frete grátis para" in line][0]
        print(f"   {doc.metadata['source']} (status={doc.metadata['status']}): {rule}")

print("\nCom make_filter() (só documentos em vigor, público customers):")
show(index.similarity_search_by_vector(shipping_vector, k=4, filter=make_filter()))

Sem filtro:
1. faq.csv                                                         Pergunta: Qual o valor mínimo para ter frete grátis? Respost
2. policies/shipping_2024.md                Frete grátis           ## Frete grátis   - **Frete grátis para compras acima de R$ 
3. policies/shipping_2024.md                Dúvidas frequentes     ## Dúvidas frequentes   ### O frete grátis vale para qualque
4. policies/shipping.md                     Frete grátis           ## Frete grátis   - **Frete grátis para compras acima de R$ 
   policies/shipping_2024.md (status=archived): - **Frete grátis para compras acima de R$ 199,00**, em todo o Brasil.
   policies/shipping.md (status=active): - **Frete grátis para compras acima de R$ 299,00** na modalidade Normal.

Com make_filter() (só documentos em vigor, público customers):
1. faq.csv                                                         Pergunta: Qual o valor mínimo para ter frete grátis? Respost
2. policies/shipping.md                     Frete gr

**Observe:** sem filtro, as duas políticas chegam juntas, com valores **diferentes** (R$ 199 na arquivada, R$ 299 na
atual), e a arquivada vem **antes**. O modelo não tem como saber qual vale, e resposta errada com citação "correta" é o
pior tipo de erro, porque parece confiável. Com o filtro, só a política em vigor.

### 6.2 Acesso: o procedimento interno

`internal/refund_approval_procedure.md` explica aos atendentes as **alçadas** de aprovação de reembolso. É útil para a
equipe, mas não deve ser mostrado ao cliente:

In [11]:
refund_question = "Como é aprovado um reembolso acima de mil reais?"
refund_vector = embeddings.embed_query(refund_question)

print("Sem filtro:")
show(index.similarity_search_by_vector(refund_vector, k=4))

print("\nCUSTOMERS_ONLY (padrão: clientes, só documentos em vigor):")
show(index.similarity_search_by_vector(refund_vector, k=4, filter=CUSTOMERS_ONLY))

print('\nmake_filter(("customers", "internal")): o assistente dos atendentes')
show(index.similarity_search_by_vector(refund_vector, k=4, filter=make_filter(("customers", "internal"))))

Sem filtro:
1. internal/refund_approval_procedure.md    Alçadas de aprovação   ## Alçadas de aprovação   | Valor do reembolso | Quem aprova
2. internal/refund_approval_procedure.md    Princípios             ## Princípios   - **Todo reembolso é aprovado por uma pessoa
3. internal/refund_approval_procedure.md    Passo a passo da análi - Avaria ou item errado: fotos da embalagem externa, da etiq
4. policies/refund.md                       Aprovação por um atend ## Aprovação por um atendente humano   **Todo reembolso pass

CUSTOMERS_ONLY (padrão: clientes, só documentos em vigor):
1. policies/refund.md                       Aprovação por um atend ## Aprovação por um atendente humano   **Todo reembolso pass
2. help_center/refund_time_by_payment_method.html                        Aprovado:  o reembolso foi liberado por um atendente e o est
3. faq.csv                                                         Pergunta: Em quanto tempo recebo o reembolso? Resposta: Depo
4. policies/refund.md     

**Observe:** sem filtro, os três primeiros trechos são do procedimento **interno**. Com `CUSTOMERS_ONLY`, a busca cai na
política pública de reembolso. E o mesmo índice atende um assistente para a equipe: basta liberar o público
`internal` no filtro.

### 6.3 A armadilha do `fetch_k`

O FAISS filtra **depois** de buscar: primeiro pega `fetch_k` candidatos (padrão 20), depois descarta os que não passam
no filtro e só então corta em `k`. Se a pergunta atrai muitos trechos proibidos, sobram menos de `k`:

In [12]:
for fetch_k in (5, 20, 50):
    docs = index.similarity_search_by_vector(refund_vector, k=4, filter=CUSTOMERS_ONLY, fetch_k=fetch_k)
    print(f"fetch_k={fetch_k:>2}: {len(docs)} trecho(s) devolvido(s)")

fetch_k= 5: 1 trecho(s) devolvido(s)
fetch_k=20: 4 trecho(s) devolvido(s)
fetch_k=50: 4 trecho(s) devolvido(s)


Com `fetch_k=5`, quatro dos cinco candidatos eram internos e só sobrou **um** trecho. Por isso a função do produto
`vector_search` usa `fetch_k = max(50, k * 5)`: busca bem mais candidatos do que precisa para que o filtro não esvazie
o resultado.

## 7. Filtro como controle de atualidade e de acesso

Por que filtrar na **busca** e não pedir no prompt ("não use documentos arquivados nem internos")?

| No prompt | No filtro da busca |
|---|---|
| um **pedido** ao modelo, que pode ser ignorado | uma **garantia** do código: o trecho nem chega ao modelo |
| o texto proibido entra no contexto (e pode vazar) | o que não entra no contexto não vaza |
| difícil de testar | função pura, testável sem LLM |

> ⚠️ Dois cuidados de produção:
> - O filtro só é tão bom quanto os **metadados**. `make_filter` assume `active` e `customers` quando a chave falta
>   (padrão "aberto"): um documento interno sem front matter apareceria para os clientes. A ingestão precisa
>   garantir esses campos.
> - Para dados realmente sensíveis ou de clientes diferentes (multi-tenant), prefira **índices separados** ou filtros
>   aplicados pelo servidor do banco vetorial, além do filtro na aplicação.

## 8. Do notebook para o projeto: [src/rag/retrieval.py](../src/rag/retrieval.py)

O produto concentra a recuperação em `src/rag/retrieval.py`. Nesta aula usamos três peças dele:

- `make_filter(audiences=("customers",), include_archived=False)`: cria a função de filtro (público permitido e, por
  padrão, só documentos em vigor);
- `CUSTOMERS_ONLY = make_filter()`: o filtro padrão do assistente dos clientes;
- `vector_search(index, query, k, metadata_filter=None)`: `similarity_search` com `fetch_k` folgado para sobrar
  candidatos depois do filtro.

Testamos os filtros com metadados de exemplo e usamos `vector_search` nas duas perguntas problemáticas da aula:

In [13]:
from src.rag.retrieval import vector_search

# Os filtros são funções puras: dá para testá-los sem índice e sem modelo.
samples = {
    "política em vigor": {"status": "active", "audience": "customers"},
    "frete de 2024": {"status": "archived", "audience": "customers"},
    "procedimento interno": {"status": "active", "audience": "internal"},
    "sem metadados": {},
}
filters = {
    "CUSTOMERS_ONLY": CUSTOMERS_ONLY,
    "equipe": make_filter(("customers", "internal")),
    "histórico": make_filter(include_archived=True),
}
print(f"{'':<22}" + "".join(f"{name:>16}" for name in filters))
for label, metadata in samples.items():
    print(f"{label:<22}" + "".join(f"{str(f(metadata)):>16}" for f in filters.values()))

# vector_search com o filtro padrão nas duas perguntas problemáticas (cada chamada calcula o embedding da consulta).
print()
for q in (shipping_question, refund_question):
    sources = [d.metadata["source"] for d in vector_search(index, q, k=4, metadata_filter=CUSTOMERS_ONLY)]
    print(f"{q}\n   {len(sources)} trechos: {sources}")

                        CUSTOMERS_ONLY          equipe       histórico
política em vigor                 True            True            True
frete de 2024                    False           False            True
procedimento interno             False            True           False
sem metadados                     True            True            True



Qual o valor mínimo para ter frete grátis?
   4 trechos: ['faq.csv', 'policies/shipping.md', 'faq.csv', 'policies/shipping.md']
Como é aprovado um reembolso acima de mil reais?
   4 trechos: ['policies/refund.md', 'help_center/refund_time_by_payment_method.html', 'faq.csv', 'policies/refund.md']


**Observe:** nenhuma fonte arquivada ou interna, e sempre `k` trechos, graças ao `fetch_k` folgado. Na próxima aula,
`vector_search` vira uma das três estratégias do `HybridRetriever`, que aplica `CUSTOMERS_ONLY` por padrão.

## Resumo

- Um **retriever** recebe uma string e devolve `list[Document]`. `index.as_retriever(...)` cria um a partir do FAISS, e
  por ser Runnable ele funciona com `invoke`, `batch` e `|`.
- `k` equilibra **chance de trazer o trecho certo** e **custo/ruído**. O projeto usa `k=4`.
- No FAISS do projeto, a nota de `similarity_search_with_score` é uma **distância** (menor é melhor);
  `cosseno = 1 - distância / 2`.
- Um **limiar fixo** de score erra para os dois lados: trechos vizinhos do assunto passam e trechos úteis ficam de fora.
  Decidir "a base responde?" é papel da geração e da avaliação de relevância.
- **MMR** troca repetição por diversidade (`fetch_k`, `lambda_mult`); diversidade demais traz irrelevância.
- **Filtros por metadados** garantem atualidade (`status`) e acesso (`audience`) antes do modelo ver o texto. O FAISS
  filtra depois de buscar: use `fetch_k` folgado.
- No projeto: `make_filter`, `CUSTOMERS_ONLY` e `vector_search` em `src/rag/retrieval.py`.

## Próxima aula

A busca vetorial entende sinônimos, mas tropeça em **códigos e nomes exatos** ("NB-002", "IPX7", "BEMVINDO10"). Na
aula 2.2 vamos somar a busca lexical **BM25**, juntar as duas com a fusão **RRF**, reordenar com o LLM e, principalmente,
**medir** se a busca híbrida realmente ajuda nesta base.